# NORTHSTAR -- Tower 23 (Web): Solace Browser + solaceagi.com QA

**Tower:** 23 -- Tower of Web
**DNA:** `web(universal) = responsive(mobile_first) x pwa(offline) x seo(discoverable) x cloud_twin(persistent) x api(documented) x cdn(fast) x a11y(wcag_aaa)`
**Floors:** 49 (343 questions, 7q/floor)
**Targets:** solace-browser, solaceagi
**Protocol:** FORECAST -> CROSS-REF -> AUDIT -> FIX -> VERIFY -> SEAL -> POSTMORTEM

This notebook probes floors 1-12 of the Tower of Web:

| Floor | Persona | Domain |
|-------|---------|--------|
| 1 | Responsive Design Expert | mobile-first responsive layout |
| 2 | PWA Expert | Progressive Web App capabilities |
| 3 | SEO Expert | search engine optimization |
| 4 | Cloud Twin Architect | browser-as-a-service cloud deployment |
| 5 | Landing Page Expert | marketing page conversion |
| 6 | Cross-Browser Tester | browser compatibility |
| 7 | API Documentation Expert | public API documentation |
| 8 | CDN Expert | content delivery and edge caching |
| 9 | HTTPS/TLS Expert | transport security |
| 10 | Web Accessibility Expert | WCAG 2.1 AA compliance for web |
| 11 | Web Performance Expert | Lighthouse 100 performance score |
| 12 | Analytics Expert | web analytics and tracking |

**Scoring:** Binary PASS/FAIL per probe. Evidence hash for tamper detection.

In [ ]:
# Cell 1: Bootstrap -- imports, paths, helper functions
# Tower 23: Web QA -- probing solace-browser for web standards compliance
import os, sys, re, json, hashlib, glob as globmod
from pathlib import Path
from datetime import datetime

NORTHSTAR = "web-qa"
NOTEBOOK_PATH = "notebooks/qa/08-web-qa.ipynb"
TOWER = 23
TOWER_NAME = "Tower of Web"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER_ROOT / "web"
SRC = BROWSER_ROOT / "src"
DATA = BROWSER_ROOT / "data" / "default"
CSS_FILE = WEB / "css" / "site.css"
JS_DIR = WEB / "js"
VENDOR_DIR = WEB / "vendor"
IMAGES_DIR = WEB / "images"
THEMES_DIR = WEB / "css" / "themes"
LOCALES_DIR = BROWSER_ROOT / "app" / "locales"

results = []

def record(probe_id, name, passed, detail=""):
    """Record a PASS/FAIL result with evidence detail."""
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    """Safely read a file, returning empty string if missing."""
    p = Path(path)
    return p.read_text(encoding="utf-8", errors="replace") if p.exists() else ""

def file_exists(path):
    """Check if file exists at path."""
    return Path(path).exists()

def file_size_kb(path):
    """Return file size in KB."""
    p = Path(path)
    return round(p.stat().st_size / 1024, 1) if p.exists() else 0

def collect_files(directory, pattern="*", recursive=True):
    """Collect all files matching pattern under directory."""
    d = Path(directory)
    if not d.exists():
        return []
    return list(d.rglob(pattern) if recursive else d.glob(pattern))

def search_in_files(directory, pattern, file_glob="*.html"):
    """Search for regex pattern in files. Return list of (filepath, match_count)."""
    hits = []
    d = Path(directory)
    if not d.exists():
        return hits
    for f in d.rglob(file_glob):
        content = read_file(f)
        matches = re.findall(pattern, content, re.IGNORECASE)
        if matches:
            hits.append((str(f.relative_to(BROWSER_ROOT)), len(matches)))
    return hits

assert BROWSER_ROOT.exists(), f"solace-browser not found at {BROWSER_ROOT}"
html_files = collect_files(WEB, "*.html", recursive=True)
print(f"Tower {TOWER}: {TOWER_NAME}")
print(f"Browser root: {BROWSER_ROOT}")
print(f"HTML files:   {len(html_files)}")
print(f"CSS file:     {CSS_FILE} ({file_size_kb(CSS_FILE)} KB)")
print(f"JS dir:       {JS_DIR} (exists={JS_DIR.exists()})")
print(f"Images dir:   {IMAGES_DIR} (exists={IMAGES_DIR.exists()})")
print("Bootstrap complete.")

In [ ]:
# ============================================================
# Floor 1: Responsive Design Expert
# Persona: mobile-first responsive layout
# Questions probed:
#   Q1: HTML validation -- proper doctype on all pages
#   Q2: lang attribute on <html> element
#   Q3: meta viewport present on all pages
#   Q4: Mobile responsiveness -- media queries in CSS
#   Q5: Flexbox/Grid used for responsive layouts
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)

# Q1: Proper DOCTYPE on all HTML pages
pages_with_doctype = 0
pages_without_doctype = []
for hf in html_files:
    content = read_file(hf)
    has_doctype = bool(re.search(r'<!DOCTYPE\s+html', content, re.IGNORECASE))
    if has_doctype:
        pages_with_doctype += 1
    else:
        pages_without_doctype.append(str(hf.relative_to(BROWSER_ROOT)))
record("F1-Q1", "All pages have <!DOCTYPE html>",
       len(pages_without_doctype) == 0,
       f"{pages_with_doctype}/{len(html_files)} have DOCTYPE. Missing: {pages_without_doctype[:5]}")

# Q2: lang attribute on <html>
pages_with_lang = 0
pages_without_lang = []
for hf in html_files:
    content = read_file(hf)
    has_lang = bool(re.search(r'<html[^>]*\slang\s*=', content, re.IGNORECASE))
    if has_lang:
        pages_with_lang += 1
    else:
        # Skip partials (they don't have <html>)
        if '<html' in content.lower():
            pages_without_lang.append(str(hf.relative_to(BROWSER_ROOT)))
record("F1-Q2", "All pages with <html> have lang attribute",
       len(pages_without_lang) == 0,
       f"{pages_with_lang} have lang attr. Missing: {pages_without_lang[:5]}")

# Q3: meta viewport on all pages
pages_with_viewport = 0
pages_without_viewport = []
for hf in html_files:
    content = read_file(hf)
    has_viewport = bool(re.search(r'<meta[^>]*name=["\']viewport["\']', content, re.IGNORECASE))
    if has_viewport:
        pages_with_viewport += 1
    else:
        # Only flag full pages (with <head>), not partials
        if '<head' in content.lower():
            pages_without_viewport.append(str(hf.relative_to(BROWSER_ROOT)))
record("F1-Q3", "All pages have meta viewport",
       len(pages_without_viewport) == 0,
       f"{pages_with_viewport} have viewport. Missing: {pages_without_viewport[:5]}")

# Q4: Media queries in CSS for responsiveness
css_content = read_file(CSS_FILE)
media_queries = re.findall(r'@media[^{]*\{', css_content)
breakpoints = set()
for mq in media_queries:
    widths = re.findall(r'(?:min|max)-width\s*:\s*([\d.]+(?:px|em|rem))', mq)
    breakpoints.update(widths)
record("F1-Q4", "CSS has responsive media queries (>=3 breakpoints)",
       len(breakpoints) >= 3,
       f"{len(media_queries)} @media rules, breakpoints: {sorted(breakpoints)[:10]}")

# Q5: Flexbox/Grid used for layouts
flex_count = len(re.findall(r'display\s*:\s*flex', css_content, re.IGNORECASE))
grid_count = len(re.findall(r'display\s*:\s*grid', css_content, re.IGNORECASE))
record("F1-Q5", "Flexbox/Grid used for responsive layouts",
       flex_count + grid_count > 5,
       f"display:flex={flex_count}, display:grid={grid_count}")

In [ ]:
# ============================================================
# Floor 2: PWA Expert
# Persona: Progressive Web App capabilities
# Questions probed:
#   Q1: Sitemap exists (sitemap.xml)
#   Q2: robots.txt exists
#   Q3: Manifest.json / webmanifest exists
#   Q4: Service worker exists
#   Q5: PWA icons at correct sizes (192, 512)
# ============================================================

# Q1: Sitemap exists
sitemap_paths = [
    WEB / "sitemap.xml",
    BROWSER_ROOT / "sitemap.xml",
    WEB / "sitemap.json",
]
sitemap_found = None
for p in sitemap_paths:
    if p.exists():
        sitemap_found = p
        break
record("F2-Q1", "sitemap.xml exists",
       sitemap_found is not None,
       f"Checked: {[str(p) for p in sitemap_paths]}, found: {sitemap_found}")

# Q2: robots.txt exists
robots_paths = [
    WEB / "robots.txt",
    BROWSER_ROOT / "robots.txt",
]
robots_found = None
for p in robots_paths:
    if p.exists():
        robots_found = p
        break
if robots_found:
    robots_content = read_file(robots_found)
    has_sitemap_ref = 'sitemap' in robots_content.lower()
else:
    has_sitemap_ref = False
record("F2-Q2", "robots.txt exists",
       robots_found is not None,
       f"Found: {robots_found}, references sitemap: {has_sitemap_ref}")

# Q3: Manifest.json for PWA
manifest_paths = [
    WEB / "manifest.json",
    WEB / "manifest.webmanifest",
    WEB / "site.webmanifest",
    BROWSER_ROOT / "manifest.json",
]
manifest_found = None
for p in manifest_paths:
    if p.exists():
        manifest_found = p
        break
# Also check HTML for manifest link
manifest_links = []
for hf in html_files:
    content = read_file(hf)
    links = re.findall(r'<link[^>]*rel=["\']manifest["\'][^>]*>', content, re.IGNORECASE)
    manifest_links.extend(links)
record("F2-Q3", "PWA manifest exists",
       manifest_found is not None or len(manifest_links) > 0,
       f"File: {manifest_found}, HTML links: {len(manifest_links)}")

# Q4: Service worker
sw_paths = [
    WEB / "sw.js", WEB / "service-worker.js", WEB / "serviceworker.js",
    BROWSER_ROOT / "sw.js", BROWSER_ROOT / "service-worker.js",
]
sw_found = None
for p in sw_paths:
    if p.exists():
        sw_found = p
        break
sw_registrations = search_in_files(WEB, r'serviceWorker\.register', "*.html")
sw_registrations += search_in_files(WEB, r'serviceWorker\.register', "*.js")
record("F2-Q4", "Service worker exists or registered",
       sw_found is not None or len(sw_registrations) > 0,
       f"SW file: {sw_found}, registrations: {sw_registrations[:3]}")

# Q5: PWA icons at correct sizes
icon_sizes_needed = [192, 512]
found_sizes = set()
for img in collect_files(IMAGES_DIR, "*.png", recursive=True):
    for size in icon_sizes_needed:
        if str(size) in img.name:
            found_sizes.add(size)
# Also check manifest for icon declarations
if manifest_found:
    manifest_content = read_file(manifest_found)
    for size in icon_sizes_needed:
        if f"{size}x{size}" in manifest_content or f"{size}" in manifest_content:
            found_sizes.add(size)
record("F2-Q5", "PWA icons at required sizes (192, 512)",
       192 in found_sizes and 512 in found_sizes,
       f"Found sizes: {found_sizes}, needed: {icon_sizes_needed}")

In [ ]:
# ============================================================
# Floor 3: SEO Expert
# Persona: search engine optimization
# Questions probed:
#   Q1: Every page has <title> tag
#   Q2: Every page has meta description
#   Q3: Open Graph tags present (og:title, og:description, og:image)
#   Q4: Semantic heading hierarchy (h1 before h2, etc.)
#   Q5: Canonical URL meta tags
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)
# Filter to full pages only (have <head>)
full_pages = [hf for hf in html_files if '<head' in read_file(hf).lower()]

# Q1: Every page has <title>
pages_with_title = 0
pages_without_title = []
for hf in full_pages:
    content = read_file(hf)
    has_title = bool(re.search(r'<title[^>]*>[^<]+</title>', content, re.IGNORECASE))
    if has_title:
        pages_with_title += 1
    else:
        pages_without_title.append(str(hf.relative_to(BROWSER_ROOT)))
record("F3-Q1", "All pages have <title>",
       len(pages_without_title) == 0,
       f"{pages_with_title}/{len(full_pages)} have title. Missing: {pages_without_title[:5]}")

# Q2: Every page has meta description
pages_with_desc = 0
pages_without_desc = []
for hf in full_pages:
    content = read_file(hf)
    has_desc = bool(re.search(r'<meta[^>]*name=["\']description["\'][^>]*content=["\'][^"\']', content, re.IGNORECASE))
    if not has_desc:
        # Also check reverse order: content before name
        has_desc = bool(re.search(r'<meta[^>]*content=["\'][^"\'][^>]*name=["\']description["\']', content, re.IGNORECASE))
    if has_desc:
        pages_with_desc += 1
    else:
        pages_without_desc.append(str(hf.relative_to(BROWSER_ROOT)))
record("F3-Q2", "All pages have meta description",
       len(pages_without_desc) == 0,
       f"{pages_with_desc}/{len(full_pages)} have description. Missing: {pages_without_desc[:5]}")

# Q3: Open Graph tags present
og_tags = {"og:title": [], "og:description": [], "og:image": []}
for hf in full_pages:
    content = read_file(hf)
    for tag in og_tags:
        if re.search(rf'property=["\']{ re.escape(tag) }["\']', content, re.IGNORECASE):
            og_tags[tag].append(str(hf.relative_to(BROWSER_ROOT)))
og_coverage = {k: len(v) for k, v in og_tags.items()}
all_og = all(len(v) > 0 for v in og_tags.values())
record("F3-Q3", "Open Graph tags present (og:title, og:description, og:image)",
       all_og,
       f"OG coverage across {len(full_pages)} pages: {og_coverage}")

# Q4: Semantic heading hierarchy (h1 present, no h2 before h1)
heading_issues = []
for hf in full_pages:
    content = read_file(hf)
    headings = re.findall(r'<(h[1-6])[^>]*>', content, re.IGNORECASE)
    if headings:
        # Check: first heading should be h1
        if headings[0].lower() != 'h1':
            heading_issues.append((str(hf.relative_to(BROWSER_ROOT)), f"first heading is {headings[0]}"))
        # Check: no skipped levels (h1 then h3 without h2)
        levels = [int(h[1]) for h in headings]
        for i in range(1, len(levels)):
            if levels[i] > levels[i-1] + 1:
                heading_issues.append((str(hf.relative_to(BROWSER_ROOT)), f"skipped from h{levels[i-1]} to h{levels[i]}"))
                break
    else:
        heading_issues.append((str(hf.relative_to(BROWSER_ROOT)), "no headings found"))
record("F3-Q4", "Semantic heading hierarchy (h1 first, no skips)",
       len(heading_issues) == 0,
       f"{len(heading_issues)} pages with heading issues: {heading_issues[:5]}")

# Q5: Canonical URLs
pages_with_canonical = 0
for hf in full_pages:
    content = read_file(hf)
    has_canonical = bool(re.search(r'<link[^>]*rel=["\']canonical["\']', content, re.IGNORECASE))
    if has_canonical:
        pages_with_canonical += 1
record("F3-Q5", "Canonical URL tags present",
       pages_with_canonical > 0,
       f"{pages_with_canonical}/{len(full_pages)} pages have rel=canonical")

In [ ]:
# ============================================================
# Floor 4: Link Integrity + Form Accessibility
# Persona: broken links and form a11y
# Questions probed:
#   Q1: No broken internal hrefs (all href targets exist as files)
#   Q2: Form accessibility -- labels for all inputs
#   Q3: Fieldsets used for grouped form controls
#   Q4: External links have rel=noopener
#   Q5: No empty href or href="#" links (meaningless navigation)
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)

# Q1: Internal link integrity check
broken_links = []
for hf in html_files:
    content = read_file(hf)
    # Find all href values
    hrefs = re.findall(r'href=["\']([^"\']*)["\'\s]', content)
    for href in hrefs:
        # Skip external, mailto, tel, javascript, anchors
        if href.startswith(('http', 'mailto:', 'tel:', 'javascript:', '#', 'data:')):
            continue
        if not href or href == '/':
            continue
        # Resolve relative path
        clean = href.split('?')[0].split('#')[0]  # remove query/hash
        if clean.startswith('/'):
            target = WEB / clean.lstrip('/')
        else:
            target = hf.parent / clean
        if not target.exists():
            # Check with .html extension
            if not (target.parent / (target.name + '.html')).exists():
                broken_links.append((str(hf.relative_to(BROWSER_ROOT)), href))
record("F4-Q1", "No broken internal links",
       len(broken_links) == 0,
       f"Found {len(broken_links)} broken internal links: {broken_links[:10]}")

# Q2: Form labels for all inputs
forms_data = []
for hf in html_files:
    content = read_file(hf)
    # Find all <input> with type != hidden, submit, button, image
    inputs = re.findall(r'<input[^>]*>', content, re.IGNORECASE)
    for inp in inputs:
        inp_type = re.search(r'type=["\']([^"\']*)["\'\s]', inp, re.IGNORECASE)
        t = inp_type.group(1).lower() if inp_type else 'text'
        if t in ('hidden', 'submit', 'button', 'image', 'reset'):
            continue
        # Check for id (which label can reference) or aria-label
        has_id = bool(re.search(r'\bid\s*=', inp, re.IGNORECASE))
        has_aria = bool(re.search(r'aria-label', inp, re.IGNORECASE))
        has_placeholder_only = bool(re.search(r'placeholder', inp, re.IGNORECASE))
        if not has_id and not has_aria:
            forms_data.append((str(hf.relative_to(BROWSER_ROOT)), inp[:60]))
    # Also check <textarea> and <select>
    for tag in ['textarea', 'select']:
        elements = re.findall(rf'<{tag}[^>]*>', content, re.IGNORECASE)
        for el in elements:
            has_id = bool(re.search(r'\bid\s*=', el, re.IGNORECASE))
            has_aria = bool(re.search(r'aria-label', el, re.IGNORECASE))
            if not has_id and not has_aria:
                forms_data.append((str(hf.relative_to(BROWSER_ROOT)), el[:60]))
record("F4-Q2", "All form inputs have labels (id or aria-label)",
       len(forms_data) == 0,
       f"{len(forms_data)} inputs without labels: {forms_data[:5]}")

# Q3: Fieldsets used for grouped controls
fieldset_count = 0
form_count = 0
for hf in html_files:
    content = read_file(hf)
    form_count += len(re.findall(r'<form[\s>]', content, re.IGNORECASE))
    fieldset_count += len(re.findall(r'<fieldset[\s>]', content, re.IGNORECASE))
record("F4-Q3", "Fieldsets used for form grouping",
       fieldset_count > 0 or form_count == 0,
       f"Forms: {form_count}, Fieldsets: {fieldset_count}")

# Q4: External links have rel=noopener
external_without_noopener = []
for hf in html_files:
    content = read_file(hf)
    ext_links = re.findall(r'<a[^>]*href=["\']https?://[^>]*>', content, re.IGNORECASE)
    for link in ext_links:
        if 'target=' in link.lower():
            has_noopener = bool(re.search(r'rel=["\'][^"\']*(noopener|noreferrer)', link, re.IGNORECASE))
            if not has_noopener:
                external_without_noopener.append((str(hf.relative_to(BROWSER_ROOT)), link[:60]))
record("F4-Q4", "External links with target have rel=noopener",
       len(external_without_noopener) == 0,
       f"{len(external_without_noopener)} external links missing noopener: {external_without_noopener[:5]}")

# Q5: No empty href or href="#" (meaningless navigation)
empty_hrefs = []
for hf in html_files:
    content = read_file(hf)
    bad = re.findall(r'<a[^>]*href=["\']\s*#?\s*["\'][^>]*>', content, re.IGNORECASE)
    for b in bad:
        # Skip if it has an onclick or role=button (intentional)
        if 'onclick' not in b.lower() and 'role="button"' not in b.lower():
            empty_hrefs.append((str(hf.relative_to(BROWSER_ROOT)), b[:60]))
record("F4-Q5", "No meaningless href='' or href='#' links",
       len(empty_hrefs) < 5,
       f"Found {len(empty_hrefs)} empty/# href links: {empty_hrefs[:5]}")

In [ ]:
# ============================================================
# Floor 5: Progressive Enhancement Expert
# Persona: works without JS for critical content
# Questions probed:
#   Q1: Critical content visible in HTML (not JS-only)
#   Q2: <noscript> fallback present
#   Q3: Navigation works without JS (links, not JS-only routing)
#   Q4: Print stylesheet exists
#   Q5: Forms have HTML5 validation (required, type, pattern)
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)
full_pages = [hf for hf in html_files if '<head' in read_file(hf).lower()]

# Q1: Critical content visible in HTML (not JS-only)
# Check if main content exists as HTML text, not injected by JS
critical_pages = ['home.html', 'download.html', 'app-store.html']
js_only_content = []
for page_name in critical_pages:
    page = WEB / page_name
    if page.exists():
        content = read_file(page)
        # Check if there's meaningful text content in the body (not just script tags)
        body_match = re.search(r'<body[^>]*>(.*)</body>', content, re.DOTALL | re.IGNORECASE)
        if body_match:
            body = body_match.group(1)
            # Strip all script/style tags
            text_only = re.sub(r'<script[^>]*>.*?</script>', '', body, flags=re.DOTALL | re.IGNORECASE)
            text_only = re.sub(r'<style[^>]*>.*?</style>', '', text_only, flags=re.DOTALL | re.IGNORECASE)
            text_only = re.sub(r'<[^>]+>', '', text_only)
            text_only = text_only.strip()
            if len(text_only) < 50:
                js_only_content.append(page_name)
record("F5-Q1", "Critical pages have HTML content (not JS-only)",
       len(js_only_content) == 0,
       f"Pages with <50 chars of non-script content: {js_only_content if js_only_content else 'none'}")

# Q2: <noscript> fallback
pages_with_noscript = 0
for hf in full_pages:
    content = read_file(hf)
    if '<noscript' in content.lower():
        pages_with_noscript += 1
record("F5-Q2", "<noscript> fallback present on some pages",
       pages_with_noscript > 0,
       f"{pages_with_noscript}/{len(full_pages)} pages have <noscript>")

# Q3: Navigation uses real <a href> links (not JS-only)
nav_links = 0
js_only_nav = 0
for hf in full_pages:
    content = read_file(hf)
    # Count <a href> in nav areas
    nav_anchors = re.findall(r'<a[^>]*href=["\'][^"\']', content, re.IGNORECASE)
    nav_links += len(nav_anchors)
    # Check for onclick-only navigation without href
    onclick_nav = re.findall(r'<(?:div|span|button)[^>]*onclick=[^>]*(?:location|navigate|window\.href)', content, re.IGNORECASE)
    js_only_nav += len(onclick_nav)
record("F5-Q3", "Navigation uses real <a href> links",
       nav_links > js_only_nav * 5,  # At least 5x more real links than JS nav
       f"<a href> links: {nav_links}, onclick-only nav: {js_only_nav}")

# Q4: Print stylesheet exists
css_content = read_file(CSS_FILE)
has_print_media = bool(re.search(r'@media\s+print', css_content, re.IGNORECASE))
# Also check for separate print.css file
print_css = WEB / "css" / "print.css"
has_print_file = print_css.exists()
# Check HTML for print stylesheet link
print_links = []
for hf in full_pages:
    content = read_file(hf)
    pl = re.findall(r'<link[^>]*media=["\']print["\']', content, re.IGNORECASE)
    print_links.extend(pl)
record("F5-Q4", "Print stylesheet exists (@media print or print.css)",
       has_print_media or has_print_file or len(print_links) > 0,
       f"@media print in site.css: {has_print_media}, print.css: {has_print_file}, print links: {len(print_links)}")

# Q5: Forms use HTML5 validation
html5_validation = 0
forms_without_validation = 0
for hf in html_files:
    content = read_file(hf)
    inputs = re.findall(r'<input[^>]*>', content, re.IGNORECASE)
    for inp in inputs:
        inp_type = re.search(r'type=["\']([^"\']*)["\'\s]', inp, re.IGNORECASE)
        t = inp_type.group(1).lower() if inp_type else 'text'
        if t in ('hidden', 'submit', 'button', 'image', 'reset'):
            continue
        has_required = 'required' in inp.lower()
        has_pattern = 'pattern=' in inp.lower()
        has_type = t in ('email', 'url', 'tel', 'number', 'date', 'time', 'search')
        if has_required or has_pattern or has_type:
            html5_validation += 1
        else:
            forms_without_validation += 1
total = html5_validation + forms_without_validation
pct = round(html5_validation / total * 100) if total > 0 else 100
record("F5-Q5", "Form inputs use HTML5 validation (>=30%)",
       pct >= 30 or total == 0,
       f"{html5_validation}/{total} inputs use HTML5 validation ({pct}%)")

In [ ]:
# ============================================================
# Floor 6: Favicon and App Icons
# Persona: branding and icon completeness
# Questions probed:
#   Q1: Favicon exists (favicon.ico, favicon.svg, or PNG)
#   Q2: Apple touch icon present
#   Q3: Multiple favicon sizes (16, 32, 48+)
#   Q4: SVG favicon for scalability
#   Q5: theme-color meta tag
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)
full_pages = [hf for hf in html_files if '<head' in read_file(hf).lower()]

# Q1: Favicon exists
favicon_paths = [
    WEB / "favicon.ico",
    WEB / "favicon.svg",
    WEB / "favicon.png",
]
favicon_found = [str(p.relative_to(BROWSER_ROOT)) for p in favicon_paths if p.exists()]
# Also check HTML for favicon links
favicon_links = []
for hf in full_pages[:5]:  # Check first 5 pages
    content = read_file(hf)
    fl = re.findall(r'<link[^>]*rel=["\'](?:icon|shortcut icon)["\'][^>]*>', content, re.IGNORECASE)
    favicon_links.extend(fl)
record("F6-Q1", "Favicon exists",
       len(favicon_found) > 0 or len(favicon_links) > 0,
       f"Favicon files: {favicon_found}, HTML links: {len(favicon_links)}")

# Q2: Apple touch icon
apple_icon = []
for hf in full_pages[:5]:
    content = read_file(hf)
    ai = re.findall(r'<link[^>]*rel=["\']apple-touch-icon["\'][^>]*>', content, re.IGNORECASE)
    apple_icon.extend(ai)
record("F6-Q2", "Apple touch icon present",
       len(apple_icon) > 0,
       f"Found {len(apple_icon)} apple-touch-icon links")

# Q3: Multiple favicon sizes
yinyang_icons = collect_files(IMAGES_DIR / "yinyang", "yinyang-logo-*.png", recursive=False)
icon_sizes = []
for ic in yinyang_icons:
    size_match = re.search(r'(\d+)', ic.name)
    if size_match:
        icon_sizes.append(int(size_match.group(1)))
has_16 = 16 in icon_sizes
has_32 = 32 in icon_sizes
has_48_plus = any(s >= 48 for s in icon_sizes)
record("F6-Q3", "Multiple favicon sizes (16, 32, 48+)",
       has_16 and has_32 and has_48_plus,
       f"Icon sizes found: {sorted(icon_sizes)}")

# Q4: SVG favicon for scalability
svg_favicon = (WEB / "favicon.svg").exists()
svg_link_found = False
for hf in full_pages[:5]:
    content = read_file(hf)
    if re.search(r'<link[^>]*type=["\']image/svg\+xml["\']', content, re.IGNORECASE):
        svg_link_found = True
        break
record("F6-Q4", "SVG favicon present",
       svg_favicon or svg_link_found,
       f"favicon.svg exists: {svg_favicon}, SVG link in HTML: {svg_link_found}")

# Q5: theme-color meta tag
theme_color_pages = 0
for hf in full_pages:
    content = read_file(hf)
    if re.search(r'<meta[^>]*name=["\']theme-color["\']', content, re.IGNORECASE):
        theme_color_pages += 1
record("F6-Q5", "theme-color meta tag present",
       theme_color_pages > 0,
       f"{theme_color_pages}/{len(full_pages)} pages have theme-color meta")

In [ ]:
# ============================================================
# Floor 7: CSP and Security Headers Expert
# Persona: Content Security Policy and HTTP security
# Questions probed:
#   Q1: Content-Security-Policy meta tag or header
#   Q2: CSP restricts unsafe-inline / unsafe-eval
#   Q3: X-Frame-Options equivalent (frame-ancestors in CSP)
#   Q4: Referrer-Policy meta tag
#   Q5: No inline event handlers (onclick, onload in HTML)
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)
full_pages = [hf for hf in html_files if '<head' in read_file(hf).lower()]

# Q1: CSP present
csp_pages = 0
csp_values = []
for hf in full_pages:
    content = read_file(hf)
    csp = re.findall(r'<meta[^>]*Content-Security-Policy[^>]*content=["\']([^"\']*)["\'\s]', content, re.IGNORECASE)
    if not csp:
        csp = re.findall(r'<meta[^>]*content=["\']([^"\']*)["\'\s][^>]*Content-Security-Policy', content, re.IGNORECASE)
    if csp:
        csp_pages += 1
        csp_values.append(csp[0][:100])
record("F7-Q1", "Content-Security-Policy present",
       csp_pages > 0,
       f"{csp_pages}/{len(full_pages)} pages have CSP. Sample: {csp_values[0][:80] if csp_values else 'none'}")

# Q2: CSP avoids unsafe-inline and unsafe-eval
unsafe_inline_pages = []
unsafe_eval_pages = []
for hf in full_pages:
    content = read_file(hf)
    csp_match = re.search(r'Content-Security-Policy["\'][^>]*content=["\']([^"\']*)["\'\s]', content, re.IGNORECASE)
    if not csp_match:
        csp_match = re.search(r'content=["\']([^"\']*)["\'\s][^>]*Content-Security-Policy', content, re.IGNORECASE)
    if csp_match:
        csp_val = csp_match.group(1)
        if "'unsafe-inline'" in csp_val:
            unsafe_inline_pages.append(str(hf.relative_to(BROWSER_ROOT)))
        if "'unsafe-eval'" in csp_val:
            unsafe_eval_pages.append(str(hf.relative_to(BROWSER_ROOT)))
record("F7-Q2", "CSP avoids unsafe-eval",
       len(unsafe_eval_pages) == 0,
       f"unsafe-inline: {len(unsafe_inline_pages)} pages, unsafe-eval: {len(unsafe_eval_pages)} pages")

# Q3: frame-ancestors in CSP (XFO equivalent)
frame_ancestors_pages = 0
for hf in full_pages:
    content = read_file(hf)
    if re.search(r'frame-ancestors', content, re.IGNORECASE):
        frame_ancestors_pages += 1
record("F7-Q3", "frame-ancestors CSP directive present",
       frame_ancestors_pages > 0,
       f"{frame_ancestors_pages}/{len(full_pages)} pages have frame-ancestors")

# Q4: Referrer-Policy
referrer_pages = 0
for hf in full_pages:
    content = read_file(hf)
    if re.search(r'<meta[^>]*name=["\']referrer["\']', content, re.IGNORECASE):
        referrer_pages += 1
    elif re.search(r'Referrer-Policy', content, re.IGNORECASE):
        referrer_pages += 1
record("F7-Q4", "Referrer-Policy meta tag present",
       referrer_pages > 0,
       f"{referrer_pages}/{len(full_pages)} pages have Referrer-Policy")

# Q5: No inline event handlers in HTML (onclick, onload, onerror, etc.)
inline_handlers = []
for hf in html_files:
    content = read_file(hf)
    handlers = re.findall(r'\s(on(?:click|load|error|submit|change|input|focus|blur|mouseover|mouseout|keydown|keyup)\s*=)', content, re.IGNORECASE)
    if handlers:
        inline_handlers.append((str(hf.relative_to(BROWSER_ROOT)), len(handlers)))
total_inline = sum(h[1] for h in inline_handlers)
record("F7-Q5", "Minimal inline event handlers (<10 total)",
       total_inline < 10,
       f"{total_inline} inline handlers across {len(inline_handlers)} files: {inline_handlers[:5]}")

In [ ]:
# ============================================================
# Floor 8: Mobile Responsiveness Expert
# Persona: mobile-first design verification
# Questions probed:
#   Q1: Touch targets minimum 44px (button/link sizing)
#   Q2: No horizontal scroll (overflow-x hidden or auto)
#   Q3: Text readable without zoom (font-size >= 16px base)
#   Q4: Mobile-friendly navigation (hamburger or collapsible)
#   Q5: Viewport units used appropriately (vh, vw)
# ============================================================

css_content = read_file(CSS_FILE)
html_files = collect_files(WEB, "*.html", recursive=True)

# Q1: Touch target sizing (44px minimum)
# Check CSS for button/link min-height or padding that achieves 44px
touch_friendly = []
button_rules = re.findall(r'(?:button|a|\.btn|\[role="button"\])[^{]*\{([^}]*)\}', css_content, re.IGNORECASE | re.DOTALL)
for rule in button_rules:
    min_h = re.findall(r'min-height\s*:\s*([\d.]+)', rule)
    padding = re.findall(r'padding\s*:[^;]*', rule)
    if min_h:
        touch_friendly.append(f"min-height: {min_h[0]}")
    if padding:
        touch_friendly.append(padding[0].strip()[:40])
record("F8-Q1", "Touch target sizing rules exist in CSS",
       len(touch_friendly) > 0,
       f"Found {len(touch_friendly)} touch-friendly sizing rules: {touch_friendly[:5]}")

# Q2: No problematic horizontal overflow
overflow_x = re.findall(r'overflow-x\s*:\s*(\w+)', css_content, re.IGNORECASE)
has_overflow_hidden = any(v.lower() in ('hidden', 'auto', 'clip') for v in overflow_x)
# Also check for max-width: 100% on images
max_width_100 = len(re.findall(r'max-width\s*:\s*100%', css_content))
record("F8-Q2", "Horizontal overflow controlled",
       has_overflow_hidden or max_width_100 > 0,
       f"overflow-x values: {overflow_x[:5]}, max-width:100% count: {max_width_100}")

# Q3: Base font-size adequate (>= 16px or 1rem)
base_font = re.findall(r'(?:html|body|:root)\s*\{[^}]*font-size\s*:\s*([^;]+);', css_content, re.IGNORECASE | re.DOTALL)
record("F8-Q3", "Base font-size set (html/body/:root)",
       len(base_font) > 0,
       f"Base font-size declarations: {[f.strip() for f in base_font][:3]}")

# Q4: Mobile navigation pattern
has_hamburger = bool(re.search(r'hamburger|menu-toggle|nav-toggle|mobile-menu|burger', css_content, re.IGNORECASE))
has_responsive_nav = bool(re.search(r'@media[^{]*\{[^}]*nav[^}]*display\s*:\s*none', css_content, re.IGNORECASE | re.DOTALL))
for hf in html_files[:5]:
    content = read_file(hf)
    if re.search(r'hamburger|menu-toggle|nav-toggle|mobile-menu', content, re.IGNORECASE):
        has_hamburger = True
        break
record("F8-Q4", "Mobile navigation pattern exists (hamburger/collapsible)",
       has_hamburger or has_responsive_nav,
       f"Hamburger: {has_hamburger}, responsive nav: {has_responsive_nav}")

# Q5: Viewport units used
vh_count = len(re.findall(r'\d+vh', css_content))
vw_count = len(re.findall(r'\d+vw', css_content))
vmin_count = len(re.findall(r'\d+vmin', css_content))
record("F8-Q5", "Viewport units used for responsive sizing",
       vh_count + vw_count + vmin_count > 0,
       f"vh: {vh_count}, vw: {vw_count}, vmin: {vmin_count}")

In [ ]:
# ============================================================
# Floor 9: Color Contrast + Accessibility Expert
# Persona: WCAG color contrast and a11y fundamentals
# Questions probed:
#   Q1: ARIA landmarks present (main, nav, banner)
#   Q2: Skip navigation link present
#   Q3: Focus styles defined (:focus, :focus-visible)
#   Q4: alt text on all images
#   Q5: Color contrast -- check CSS for color/background pairs
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)
css_content = read_file(CSS_FILE)

# Q1: ARIA landmarks
landmark_usage = {"main": 0, "nav": 0, "banner": 0, "contentinfo": 0}
for hf in html_files:
    content = read_file(hf)
    for landmark in landmark_usage:
        # Check both semantic HTML and role attribute
        if landmark == "main":
            if re.search(r'<main[\s>]|role=["\']main["\']', content, re.IGNORECASE):
                landmark_usage[landmark] += 1
        elif landmark == "nav":
            if re.search(r'<nav[\s>]|role=["\']navigation["\']', content, re.IGNORECASE):
                landmark_usage[landmark] += 1
        elif landmark == "banner":
            if re.search(r'<header[\s>]|role=["\']banner["\']', content, re.IGNORECASE):
                landmark_usage[landmark] += 1
        elif landmark == "contentinfo":
            if re.search(r'<footer[\s>]|role=["\']contentinfo["\']', content, re.IGNORECASE):
                landmark_usage[landmark] += 1
has_landmarks = all(v > 0 for v in landmark_usage.values())
record("F9-Q1", "ARIA landmarks present (main, nav, header, footer)",
       sum(v > 0 for v in landmark_usage.values()) >= 3,
       f"Landmark usage: {landmark_usage}")

# Q2: Skip navigation link
skip_nav = 0
for hf in html_files:
    content = read_file(hf)
    if re.search(r'skip.*nav|skip.*content|skiplink|skip-link', content, re.IGNORECASE):
        skip_nav += 1
record("F9-Q2", "Skip navigation link present",
       skip_nav > 0,
       f"{skip_nav} pages have skip navigation")

# Q3: Focus styles defined
focus_rules = re.findall(r':focus(?:-visible|-within)?[^{]*\{', css_content)
focus_outline = bool(re.search(r':focus[^{]*\{[^}]*(?:outline|box-shadow|border)', css_content, re.IGNORECASE | re.DOTALL))
record("F9-Q3", "Focus styles defined (:focus/:focus-visible)",
       len(focus_rules) > 0,
       f"Focus rules: {len(focus_rules)}, has outline/shadow: {focus_outline}")

# Q4: Alt text on all images
images_with_alt = 0
images_without_alt = []
for hf in html_files:
    content = read_file(hf)
    img_tags = re.findall(r'<img[^>]*>', content, re.IGNORECASE)
    for img in img_tags:
        has_alt = bool(re.search(r'\balt\s*=', img, re.IGNORECASE))
        if has_alt:
            images_with_alt += 1
        else:
            images_without_alt.append((str(hf.relative_to(BROWSER_ROOT)), img[:50]))
total_imgs = images_with_alt + len(images_without_alt)
alt_pct = round(images_with_alt / total_imgs * 100) if total_imgs > 0 else 100
record("F9-Q4", "All images have alt attribute (>=90%)",
       alt_pct >= 90,
       f"{images_with_alt}/{total_imgs} ({alt_pct}%) have alt. Missing: {images_without_alt[:5]}")

# Q5: Color contrast -- check for very light colors on white or very dark on dark
# Heuristic: look for color declarations with very light grays
light_text = re.findall(r'color\s*:\s*#(?:eee|ddd|ccc|bbb|aaa|f[0-9a-f]{5})', css_content, re.IGNORECASE)
dark_bg_dark_text = re.findall(r'background(?:-color)?\s*:\s*#(?:000|111|222|333)[^;]*', css_content, re.IGNORECASE)
# Check if theme files define contrast properly
theme_contrast = 0
for tf in collect_files(THEMES_DIR, "*.css"):
    tc = read_file(tf)
    if re.search(r'--(?:text|fg|foreground)', tc, re.IGNORECASE) and re.search(r'--(?:bg|background)', tc, re.IGNORECASE):
        theme_contrast += 1
record("F9-Q5", "Color contrast strategy (theme vars define fg/bg pairs)",
       theme_contrast > 0,
       f"Themes with fg/bg custom props: {theme_contrast}, light text colors in CSS: {len(light_text)}")

In [ ]:
# ============================================================
# Floor 10: Semantic HTML Expert
# Persona: proper heading hierarchy and semantic markup
# Questions probed:
#   Q1: Exactly one <h1> per page
#   Q2: Semantic elements used (article, section, aside, main)
#   Q3: Lists used for navigation (<ul>/<ol> in <nav>)
#   Q4: Tables have proper headers (<th>, scope, caption)
#   Q5: Buttons vs links used correctly (action vs navigation)
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)
full_pages = [hf for hf in html_files if '<head' in read_file(hf).lower()]

# Q1: Exactly one <h1> per page
h1_issues = []
for hf in full_pages:
    content = read_file(hf)
    h1_count = len(re.findall(r'<h1[\s>]', content, re.IGNORECASE))
    if h1_count != 1:
        h1_issues.append((str(hf.relative_to(BROWSER_ROOT)), h1_count))
record("F10-Q1", "Exactly one <h1> per page",
       len(h1_issues) == 0,
       f"{len(h1_issues)} pages with wrong h1 count: {h1_issues[:5]}")

# Q2: Semantic elements used
semantic_tags = {'article': 0, 'section': 0, 'aside': 0, 'main': 0, 'nav': 0, 'header': 0, 'footer': 0}
for hf in html_files:
    content = read_file(hf)
    for tag in semantic_tags:
        if re.search(rf'<{tag}[\s>]', content, re.IGNORECASE):
            semantic_tags[tag] += 1
tags_used = sum(1 for v in semantic_tags.values() if v > 0)
record("F10-Q2", "Semantic HTML elements used (>=4 types)",
       tags_used >= 4,
       f"Semantic tags: {semantic_tags}")

# Q3: Navigation uses lists
nav_with_lists = 0
nav_without_lists = 0
for hf in html_files:
    content = read_file(hf)
    navs = re.findall(r'<nav[^>]*>(.*?)</nav>', content, re.DOTALL | re.IGNORECASE)
    for nav in navs:
        if re.search(r'<ul|<ol', nav, re.IGNORECASE):
            nav_with_lists += 1
        else:
            nav_without_lists += 1
record("F10-Q3", "Navigation uses <ul>/<ol> lists",
       nav_with_lists >= nav_without_lists or (nav_with_lists + nav_without_lists) == 0,
       f"Nav with lists: {nav_with_lists}, without: {nav_without_lists}")

# Q4: Tables have proper markup
table_issues = []
for hf in html_files:
    content = read_file(hf)
    tables = re.findall(r'<table[^>]*>(.*?)</table>', content, re.DOTALL | re.IGNORECASE)
    for table in tables:
        has_th = bool(re.search(r'<th[\s>]', table, re.IGNORECASE))
        has_caption = bool(re.search(r'<caption', table, re.IGNORECASE))
        if not has_th:
            table_issues.append((str(hf.relative_to(BROWSER_ROOT)), "no <th>"))
record("F10-Q4", "Tables have proper headers (<th>)",
       len(table_issues) == 0,
       f"{len(table_issues)} tables without <th>: {table_issues[:5]}")

# Q5: Buttons vs links used correctly
button_as_link = []
for hf in html_files:
    content = read_file(hf)
    # Buttons with href-like behavior (should be <a>)
    bad_buttons = re.findall(r'<button[^>]*onclick=["\'][^"\']*(location|navigate|window\.href|window\.open)[^"\']', content, re.IGNORECASE)
    if bad_buttons:
        button_as_link.append((str(hf.relative_to(BROWSER_ROOT)), len(bad_buttons)))
    # Links styled as buttons without role=button
    link_buttons = re.findall(r'<a[^>]*class=["\'][^"\']*(btn|button)[^"\'][^>]*>', content, re.IGNORECASE)
    # These are OK as long as they navigate somewhere
record("F10-Q5", "Buttons and links used correctly",
       len(button_as_link) == 0,
       f"Buttons used for navigation: {button_as_link if button_as_link else 'none'}")

In [ ]:
# ============================================================
# Floor 11: Web Performance Expert (Lighthouse-style)
# Persona: Lighthouse 100 performance score factors
# Questions probed:
#   Q1: Scripts deferred/async (no render-blocking JS)
#   Q2: CSS minification check (no excessive whitespace)
#   Q3: JS minification check for vendor files
#   Q4: Efficient CSS selectors (no overly complex selectors)
#   Q5: Resource count per page (ideal: <25 requests)
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)

# Q1: Scripts deferred/async (covered in perf but different angle here)
blocking_scripts = []
for hf in html_files:
    content = read_file(hf)
    scripts = re.findall(r'<script[^>]*src=[^>]*>', content, re.IGNORECASE)
    for s in scripts:
        if 'defer' not in s.lower() and 'async' not in s.lower():
            src = re.search(r'src=["\']([^"\']*)["\'\s]', s)
            blocking_scripts.append(src.group(1) if src else 'unknown')
record("F11-Q1", "No render-blocking scripts (all defer/async)",
       len(blocking_scripts) == 0,
       f"Blocking scripts: {list(set(blocking_scripts))[:5]}")

# Q2: CSS minification check (ratio of whitespace to content)
css_content = read_file(CSS_FILE)
css_bytes = len(css_content.encode('utf-8'))
# Count significant whitespace (blank lines, multiple spaces)
blank_lines = len(re.findall(r'\n\s*\n', css_content))
multi_spaces = len(re.findall(r'  +', css_content))
comment_bytes = sum(len(c) for c in re.findall(r'/\*.*?\*/', css_content, re.DOTALL))
waste_ratio = round((comment_bytes + blank_lines * 2) / css_bytes * 100, 1) if css_bytes > 0 else 0
record("F11-Q2", "CSS reasonably compact (waste <20%)",
       waste_ratio < 20,
       f"CSS: {round(css_bytes/1024, 1)}KB, comments: {round(comment_bytes/1024, 1)}KB, blank lines: {blank_lines}, waste ratio: {waste_ratio}%")

# Q3: Vendor JS minified
vendor_files = collect_files(VENDOR_DIR, "*.js", recursive=True)
min_count = sum(1 for f in vendor_files if '.min.' in f.name)
non_min = [f.name for f in vendor_files if '.min.' not in f.name]
record("F11-Q3", "Vendor JS files are minified (>=50%)",
       min_count >= len(vendor_files) / 2 if vendor_files else True,
       f"Minified: {min_count}/{len(vendor_files)}, non-minified: {non_min[:5]}")

# Q4: No overly complex CSS selectors (>5 parts)
css_rules = re.findall(r'([^{}]+)\{', css_content)
complex_selectors = []
for rule in css_rules:
    for selector in rule.split(','):
        sel = selector.strip()
        # Skip @-rules, empty, and CSS comments
        if sel.startswith('@') or not sel or sel.startswith('/*') or sel.startswith('*') or '===' in sel:
            continue
        # Skip lines that are clearly comments (contain only non-selector chars)
        if re.match(r'^[\s/*=\-]+$', sel):
            continue
        parts = sel.split()
        if len(parts) > 5:
            complex_selectors.append(sel[:60])
record("F11-Q4", "No overly complex CSS selectors (>5 parts, <5 instances)",
       len(complex_selectors) < 5,
       f"Found {len(complex_selectors)} complex selectors: {complex_selectors[:3]}")

# Q5: Resource count per page (estimate from <link> and <script> tags)
max_resources = 0
max_resource_page = ""
for hf in html_files:
    content = read_file(hf)
    scripts = len(re.findall(r'<script[^>]*src=', content, re.IGNORECASE))
    css_links = len(re.findall(r'<link[^>]*stylesheet', content, re.IGNORECASE))
    images = len(re.findall(r'<img[^>]*src=', content, re.IGNORECASE))
    total = scripts + css_links + images
    if total > max_resources:
        max_resources = total
        max_resource_page = str(hf.relative_to(BROWSER_ROOT))
record("F11-Q5", "Resource count per page < 30",
       max_resources < 30,
       f"Max resources: {max_resources} on {max_resource_page}")

In [ ]:
# ============================================================
# Floor 12: Analytics Expert
# Persona: privacy-respecting analytics
# Questions probed:
#   Q1: No Google Analytics (GA4) tracking code
#   Q2: No third-party tracking pixels
#   Q3: Cookie-free analytics possible
#   Q4: GDPR-friendly (no tracking without consent)
#   Q5: Error tracking (client-side error reporting)
# ============================================================

html_files = collect_files(WEB, "*.html", recursive=True)
js_files = collect_files(WEB, "*.js", recursive=True)

# Q1: No Google Analytics
ga_hits = []
for hf in html_files:
    content = read_file(hf)
    ga_patterns = re.findall(r'(?:google-analytics\.com|googletagmanager\.com|gtag\s*\(|UA-\d{4,}-\d|G-[A-Z0-9]{10,})', content)
    if ga_patterns:
        ga_hits.append((str(hf.relative_to(BROWSER_ROOT)), ga_patterns))
for f in js_files:
    content = read_file(f)
    ga_patterns = re.findall(r'(?:google-analytics\.com|googletagmanager\.com|gtag\s*\(|UA-\d{4,}-\d|G-[A-Z0-9]{10,})', content)
    if ga_patterns:
        ga_hits.append((str(f.relative_to(BROWSER_ROOT)), ga_patterns))
record("F12-Q1", "No Google Analytics tracking code",
       len(ga_hits) == 0,
       f"GA references found in: {ga_hits if ga_hits else 'none'}")

# Q2: No third-party tracking pixels
tracking_pixels = []
for hf in html_files:
    content = read_file(hf)
    pixels = re.findall(r'(?:facebook\.com/tr|pixel|hotjar|mixpanel|segment\.com|amplitude|heap\.io)', content, re.IGNORECASE)
    if pixels:
        tracking_pixels.append((str(hf.relative_to(BROWSER_ROOT)), pixels))
record("F12-Q2", "No third-party tracking pixels",
       len(tracking_pixels) == 0,
       f"Tracking pixels: {tracking_pixels if tracking_pixels else 'none'}")

# Q3: Cookie-free design (no document.cookie writes in JS)
cookie_writes = []
for f in js_files:
    content = read_file(f)
    writes = re.findall(r'document\.cookie\s*=', content)
    if writes:
        cookie_writes.append((str(f.relative_to(BROWSER_ROOT)), len(writes)))
record("F12-Q3", "Cookie-free design (no document.cookie writes)",
       len(cookie_writes) == 0,
       f"Cookie writes: {cookie_writes if cookie_writes else 'none'}")

# Q4: GDPR-friendly (no auto-tracking scripts)
auto_track = []
for hf in html_files:
    content = read_file(hf)
    trackers = re.findall(r'(?:beacon|tracking|analytics)\.js', content, re.IGNORECASE)
    if trackers:
        auto_track.append((str(hf.relative_to(BROWSER_ROOT)), trackers))
record("F12-Q4", "No auto-tracking scripts loaded",
       len(auto_track) == 0,
       f"Auto-tracking scripts: {auto_track if auto_track else 'none'}")

# Q5: Client-side error reporting present
error_handling = 0
for f in js_files:
    content = read_file(f)
    if re.search(r'window\.onerror|window\.addEventListener\(["\']error|window\.addEventListener\(["\']unhandledrejection', content, re.IGNORECASE):
        error_handling += 1
record("F12-Q5", "Client-side error handling present",
       error_handling > 0,
       f"{error_handling} JS files with global error handlers")

In [ ]:
# ============================================================
# EVIDENCE SUMMARY -- Tower 23: Web x Solace Browser
# ============================================================

passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
finding_rate = (failed / total * 100) if total > 0 else 0

# Group by floor
floor_summary = {}
for r in results:
    floor = r["id"].split("-")[0]  # e.g. "F1"
    if floor not in floor_summary:
        floor_summary[floor] = {"passed": 0, "failed": 0, "total": 0}
    floor_summary[floor]["total"] += 1
    if r["status"] == "PASS":
        floor_summary[floor]["passed"] += 1
    else:
        floor_summary[floor]["failed"] += 1

floor_labels = {
    "F1": "Responsive Design", "F2": "PWA / Sitemap / Robots", "F3": "SEO",
    "F4": "Link Integrity / Forms", "F5": "Progressive Enhancement", "F6": "Favicon / Icons",
    "F7": "CSP / Security Headers", "F8": "Mobile Responsiveness", "F9": "Accessibility / Contrast",
    "F10": "Semantic HTML", "F11": "Web Performance (Lighthouse)", "F12": "Analytics / Privacy",
}

summary = {
    "surface": NORTHSTAR,
    "tower": TOWER,
    "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total,
    "passed": passed,
    "failed": failed,
    "finding_rate": round(finding_rate, 1),
    "score": round(passed / total * 100, 1) if total > 0 else 0,
    "min_score": MIN_SCORE,
    "converged": finding_rate < 5,
    "floor_scores": {
        f"{k} ({floor_labels.get(k, '?')})": f"{v['passed']}/{v['total']}"
        for k, v in sorted(floor_summary.items())
    },
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 70)
print(f"TOWER {TOWER}: {TOWER_NAME} -- SOLACE BROWSER QA")
print("=" * 70)
print(f"Total probes:  {total}")
print(f"Passed:        {passed}")
print(f"Failed:        {failed}")
print(f"Score:         {summary['score']}%")
print(f"Finding rate:  {finding_rate:.1f}%")
print(f"Converged:     {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
print()
print("FLOOR BREAKDOWN:")
for k, v in sorted(floor_summary.items()):
    label = floor_labels.get(k, "?")
    pct = round(v["passed"] / v["total"] * 100) if v["total"] > 0 else 0
    bar = "#" * (pct // 5) + "." * (20 - pct // 5)
    print(f"  {k:4s} {label:32s} {v['passed']}/{v['total']} [{bar}] {pct}%")

print()
if summary["findings"]:
    print(f"FINDINGS ({len(summary['findings'])}):\n")
    for f in summary["findings"]:
        print(f"  FAIL: {f['id']} {f['name']}")
        if f['detail']:
            print(f"        {f['detail'][:120]}")
else:
    print("ALL PROBES PASSED.")

print(f"\n--- Tower {TOWER} complete. Evidence sealed. ---")